# 🎯 Next Best Offer (Random Forest Multiclase)
> **Caso de negocio:** Financiera Efectiva · Cross-sell de productos financieros
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Financiera Efectiva tiene 6 productos para ofrecer a su base de clientes (Seguro vehicular, Fondo de inversión, Préstamo hipotecario, CTS digital, Tarjeta de débito, SOAT premium). Hoy los asesores de cross-sell ofrecen **el mismo producto "estrella" a todos los clientes**, sin considerar el perfil de cada uno.

**Objetivo:** Predecir cuál es el siguiente mejor producto (Next Best Offer) para cada cliente, según su perfil de productos actuales, antigüedad y saldo.

**KPIs de éxito:**
- Top-2 Accuracy >= 0.70 (el producto que el cliente realmente adquiere está entre las 2 mejores recomendaciones)
- F1-macro >= 0.50 (desempeño balanceado entre los 6 productos)
- Accuracy >= 3x el azar (1/6 ≈ 16.7%)

**Algoritmo:** Random Forest multiclase (probabilidad por producto + importancia de variables)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q shap plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import shap

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Clasificación multiclase supervisada',
    'target': 'producto_adquirido (6 productos posibles)',
    'ventana_prediccion': 'siguiente contacto comercial (call center / app)',
    'features': [
        'tiene_tarjeta   → 1 si el cliente ya tiene tarjeta de crédito',
        'tiene_cuenta    → 1 si el cliente tiene cuenta de ahorros activa',
        'tiene_seguro    → 1 si el cliente ya tiene algún seguro',
        'tiene_prestamo  → 1 si el cliente tiene un préstamo vigente',
        'antiguedad_m    → antigüedad como cliente (meses)',
        'saldo_k         → saldo promedio en cuentas (miles de S/.)',
    ],
    'criterios_aceptacion': {
        'Top-2 Accuracy': '>= 0.70',
        'F1-macro':       '>= 0.50',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura realista
# En producción: reemplazar con datos del core bancario + CRM (BigQuery)
N = 1200

prods = ['Seguro vehicular', 'Fondo inversion', 'Prestamo hipotecario',
         'CTS digital', 'Tarjeta debito', 'SOAT premium']

rows = []
for i in range(N):
    tc = int(np.random.random() > 0.4)
    ca = int(np.random.random() > 0.5)
    seg = int(np.random.random() > 0.7)
    pr = int(np.random.random() > 0.6)
    ant = np.random.randint(1, 61)
    saldo = round(np.random.uniform(1, 50), 1)
    segmento = 'Alto' if saldo > 20 else ('Medio' if saldo > 10 else 'Bajo')

    # DGP: score "oculto" por producto según el perfil del cliente
    scores = [tc*0.6+0.1, saldo/50*0.8+0.1,
              (ant/60)*0.5+(pr*0.3), ca*0.5+0.2,
              (1-ca)*0.6+0.1, tc*0.3+0.1]
    scores = [s + np.random.uniform(-0.05, 0.05) for s in scores]
    prod = prods[np.argmax(scores)]

    rows.append({'cliente_id': i+1, 'tiene_tarjeta': tc, 'tiene_cuenta': ca,
                  'tiene_seguro': seg, 'tiene_prestamo': pr,
                  'antiguedad_m': ant, 'saldo_k': saldo,
                  'segmento': segmento, 'producto_adquirido': prod})

df = pd.DataFrame(rows)

print(f'Dataset: {df.shape}')
print(f'\nDistribución del target:')
print(df['producto_adquirido'].value_counts(normalize=True).map('{:.1%}'.format))
df.head()

In [ ]:
# Distribución del producto adquirido por segmento de saldo
fig = px.histogram(df, x='producto_adquirido', color='segmento', barmode='group',
                    title='Producto adquirido por segmento de saldo',
                    color_discrete_map={'Alto': '#0d9488', 'Medio': '#d97706', 'Bajo': '#c0392b'},
                    category_orders={'segmento': ['Bajo', 'Medio', 'Alto']})
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis_title=None, yaxis_title='Clientes')
fig.show()

In [ ]:
# Distribución de antigüedad y saldo
fig = make_subplots(rows=1, cols=2, subplot_titles=['Antigüedad (meses)', 'Saldo promedio (k S/.)'])
fig.add_trace(go.Histogram(x=df['antiguedad_m'], marker_color='#1a4c8c', nbinsx=20, showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df['saldo_k'], marker_color='#7c3aed', nbinsx=20, showlegend=False), row=1, col=2)
fig.update_layout(height=350, title='Distribución de variables numéricas',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Tenencia de productos actuales por producto recomendado
flags = ['tiene_tarjeta', 'tiene_cuenta', 'tiene_seguro', 'tiene_prestamo']
tenencia = df.groupby('producto_adquirido')[flags].mean().round(2)

fig = px.imshow(tenencia, text_auto=True, color_continuous_scale='teal',
                title='Tenencia promedio de productos actuales, por producto adquirido')
fig.update_layout(height=400, paper_bgcolor='white')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['tiene_tarjeta', 'tiene_cuenta', 'tiene_seguro', 'tiene_prestamo', 'antiguedad_m', 'saldo_k']
TARGET   = 'producto_adquirido'

le = LabelEncoder()
X = df[FEATURES].fillna(0)
y = le.fit_transform(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Clases: {dict(enumerate(le.classes_))}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Random Forest multiclase: una probabilidad por producto
model = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
model.fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
cv_f1  = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro')

print(f'CV Accuracy: {cv_acc.mean():.3f} ± {cv_acc.std():.3f}')
print(f'CV F1-macro: {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# Importancia de variables
importancia = pd.DataFrame({
    'feature': FEATURES,
    'importancia': model.feature_importances_
}).sort_values('importancia', ascending=True)

fig = go.Figure(go.Bar(
    x=importancia['importancia'], y=importancia['feature'], orientation='h',
    marker_color='#1a4c8c',
    text=importancia['importancia'].map('{:.3f}'.format), textposition='outside'
))
fig.update_layout(title='Importancia de variables (Random Forest)',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0'), yaxis=dict(showgrid=False))
fig.show()

## 5️⃣ Fase 5 — Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

# Top-2 accuracy: el producto real está entre las 2 probabilidades más altas
top2_correct = sum(y_test[i] in np.argsort(y_prob[i])[-2:] for i in range(len(y_test)))
top2_acc = top2_correct / len(y_test)

metrics = {'Accuracy': acc, 'F1-macro': f1_macro, 'Top-2 Accuracy': top2_acc, 'Azar (1/6)': 1/6}
criterios = {'Top-2 Accuracy': 0.70, 'F1-macro': 0.50}

print('=== MÉTRICAS EN TEST SET ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ NECESITA MEJORA (umbral {umbral:.2f})'
    print(f'  {k:16s}: {v:.4f}  {estado}')

In [ ]:
# Matriz de confusión multiclase
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45)
ax.set_title('Matriz de Confusión — Next Best Offer')
plt.tight_layout()
plt.show()

In [ ]:
# Probabilidad media por producto (test set)
probs_mean = y_prob.mean(axis=0)
prob_df = pd.DataFrame({'producto': le.classes_, 'prob': probs_mean}).sort_values('prob', ascending=True)

colors = ['#c0392b' if i == len(prob_df)-1 else '#1a4c8c' if i >= len(prob_df)-3 else '#6b7280'
          for i in range(len(prob_df))]

fig = go.Figure(go.Bar(
    x=prob_df['prob'], y=prob_df['producto'], orientation='h',
    marker_color=colors,
    text=prob_df['prob'].map('{:.1%}'.format), textposition='outside'
))
fig.update_layout(title='Probabilidad media por producto (test set)',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0', tickformat='.0%'), yaxis=dict(showgrid=False))
fig.show()

In [ ]:
# SHAP — explicación local: ¿por qué el modelo recomienda este producto a este cliente?
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

idx = 0
pred_idx = int(y_pred[idx])
print(f'Cliente test #{idx} → producto recomendado: {le.classes_[pred_idx]}')
print(f'Perfil: {X_test.iloc[idx].to_dict()}')

# shap >= 0.45 devuelve un array (n_samples, n_features, n_clases);
# versiones previas devuelven una lista de arrays por clase
if isinstance(shap_values, list):
    sv = shap_values[pred_idx][idx]
    base_value = explainer.expected_value[pred_idx]
else:
    sv = shap_values[idx, :, pred_idx]
    base_value = explainer.expected_value[pred_idx]

shap.waterfall_plot(
    shap.Explanation(
        values=sv,
        base_values=base_value,
        data=X_test.iloc[idx],
        feature_names=FEATURES
    ),
    show=False
)
plt.title(f'¿Por qué se recomienda "{le.classes_[pred_idx]}"?')
plt.tight_layout()
plt.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Score de toda la base: producto recomendado + probabilidad
X_all = df[FEATURES].fillna(0)
probs_all = model.predict_proba(X_all)
pred_all = probs_all.argmax(axis=1)

df['producto_recomendado'] = le.inverse_transform(pred_all)
df['prob_recomendacion'] = probs_all.max(axis=1).round(3)

print('Distribución de recomendaciones para toda la base:')
print(df['producto_recomendado'].value_counts(normalize=True).map('{:.1%}'.format))
df[['cliente_id', 'segmento', 'producto_recomendado', 'prob_recomendacion']].head(10)

In [ ]:
# Guardar outputs
df[['cliente_id', 'segmento', 'producto_recomendado', 'prob_recomendacion']].to_csv(
    'next_best_offer_efectiva.csv', index=False)
print('Archivo next_best_offer_efectiva.csv guardado ✓')

import joblib
joblib.dump({'model': model, 'label_encoder': le, 'features': FEATURES},
            'modelo_nbo_efectiva.pkl')
print('Modelo modelo_nbo_efectiva.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
n_clientes = len(df)
prod_top = df['producto_recomendado'].value_counts().idxmax()
prob_alta = (df['prob_recomendacion'] >= 0.5).sum()

print('=== RESUMEN EJECUTIVO ===')
print(f'Clientes analizados:                {n_clientes:,}')
print(f'Top-2 Accuracy del modelo:          {top2_acc:.1%}')
print(f'Producto más recomendado:           {prod_top}')
print(f'Clientes con prob. >= 50%:          {prob_alta:,} ({prob_alta/n_clientes:.0%})')
print(f'\nArquitectura de producción:')
print('  Core bancario + CRM → BigQuery → score semanal de NBO → cola priorizada de cross-sell (call center / app)')